### Pre-processing 
This involves: 
* reading the Excel file
* dropping Nan rows (withdrawn, void, absent in work roles).
* reformat labels: some labels are formatted as 4,5. Others are labeled as 0.2. with/without spaces. Turn them into one-hot encodings. 
* drop knowledge unit column.  Knowledge units vs knowledge areas. The KAs may contain multiple knowledge units. Knowledge units are out of the scope of this project right now, but I should think of a way to number/label them. 
* convert to Pytorch tensor. Drop KD numbers column and headers before converting to Pytorch array. we don't need the KD numbers for training, just for interpreting the results. 

In [85]:
# import necessary packages
import pandas as pd

# read Excel file as Pandas df
path = "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/Mappings_between_CSEC2017_and_the_NICE_Framework.xlsx"

df = pd.read_excel(path, sheet_name='Mapping and KD-Weight')
print(df.columns)
print(df.index)

# drop all Nan rows (withdrawn, void, absent in work roles). 
df.dropna(axis=0, inplace=True) #0 = rows 
#print(df)

Index(['knowledge descriptions(NICE Framework)', 'Knowledge Units (CSEC2017)',
       ' Knowledge areas (CSEC2017)'],
      dtype='object')
RangeIndex(start=0, stop=630, step=1)


In [86]:
# some labels are formatted as 4,5. Others are labeled as 0.2. with/without spaces. 
# reformat so they are all the same format. 
def df_to_onehot(df, column=' Knowledge areas (CSEC2017)'): 
    # find all unique labels
    all_labels = set(label for row in df[column] for label in row.split(','))
    all_labels = list(all_labels)
    all_labels.sort()
    # 
    print(all_labels)
    # initialize one-hot encoding coluns with 0. 
    for label in all_labels: 
        df[label] = 0 

    # set 1 for all indicated labels in the specified column. 
    for index, row in df.iterrows(): 
        labels = row[column].split(',')
        for label in labels: 
            df.at[index,label] = 1 

def format_knowledge_areas(value): 
    # turn value into string for easier processing 
    value = str(value)
    # replace periods with commas 
    value = value.replace('.', ',') 
    # remove any extra spaces 
    value = value.replace(' ','')
    return value 
df[' Knowledge areas (CSEC2017)'] = df[' Knowledge areas (CSEC2017)'].apply(format_knowledge_areas)
df_to_onehot(df)
print(df)

['0', '1', '2', '3', '4', '5', '6', '7', '8']
    knowledge descriptions(NICE Framework)  \
0                                    K0001   
1                                    K0002   
2                                    K0003   
3                                    K0004   
4                                    K0005   
..                                     ...   
613                                  K0614   
614                                  K0615   
621                                  K0622   
623                                  K0624   
627                                  K0628   

                            Knowledge Units (CSEC2017)  \
0                                 Network Architecture   
1                                      Risk Management   
2       Cyber Law, Cyber Ethics, Cyber Policy, Privacy   
3     Fundamental Principles, System Thinking, Privacy   
4    Cyberspace Practice, Component Design, Network...   
..                                                 ..

In [87]:
# extract KD index column for result evaluation. 
KD_index = df['knowledge descriptions(NICE Framework)']
print(KD_index)
# drop KD index, KU and original KA columns. 
df = df.drop(columns=[' Knowledge areas (CSEC2017)','Knowledge Units (CSEC2017)'])

0      K0001
1      K0002
2      K0003
3      K0004
4      K0005
       ...  
613    K0614
614    K0615
621    K0622
623    K0624
627    K0628
Name: knowledge descriptions(NICE Framework), Length: 576, dtype: object


In [88]:
print(df)

    knowledge descriptions(NICE Framework)  0  1  2  3  4  5  6  7  8
0                                    K0001  0  0  0  0  1  0  0  0  0
1                                    K0002  0  0  0  0  0  0  0  1  0
2                                    K0003  0  0  0  0  0  0  0  0  1
3                                    K0004  0  0  1  0  0  1  0  0  1
4                                    K0005  1  0  0  1  1  1  0  0  0
..                                     ... .. .. .. .. .. .. .. .. ..
613                                  K0614  1  0  0  0  1  0  0  0  0
614                                  K0615  0  0  0  0  0  0  1  0  0
621                                  K0622  0  0  0  0  0  0  0  1  0
623                                  K0624  1  0  1  0  0  0  0  0  0
627                                  K0628  1  0  0  0  0  0  0  0  0

[576 rows x 10 columns]


In [89]:
# Add the actual content/generate another dataframe with the content. 
# read Excel file as Pandas df
path = "/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu/data/NICE Framework Components Mapping - 2017 to v1.0.0 March 2024.xlsx"

des = pd.read_excel(path, sheet_name='TKS Statements Mapping')
print(des.columns)
print(des.index)


Index(['KSAT ID', 'Statement Description', 'New', 'Withdrawn',
       'Replacement Statements (if applicable)'],
      dtype='object')
RangeIndex(start=0, stop=4344, step=1)


In [90]:
#rows_k = des[des.apply(lambda row: row.astype(str).str.startswith('K').any(), axis=1)]
rows_k = des['KSAT ID'].str.startswith('K', na=False)
#rows_k.dropna(axis=0, inplace=True)
kds = des[rows_k]
print(rows_k[:])
print(rows_k[1270])
print(kds)
# keep only the KSAT ID column and the statement descriptoin. 


0       False
1       False
2       False
3       False
4       False
        ...  
4339    False
4340    False
4341    False
4342    False
4343    False
Name: KSAT ID, Length: 4344, dtype: bool
True
     KSAT ID                              Statement Description    New  \
177    K0001  Knowledge of computer networking concepts and ...  False   
178    K0002  Knowledge of risk management processes (e.g., ...  False   
179    K0003  Knowledge of laws, regulations, policies, and ...  False   
180    K0004  Knowledge of cybersecurity and privacy principles  False   
181    K0005     Knowledge of cyber threats and vulnerabilities  False   
...      ...                                                ...    ...   
1431   K1271  Knowledge of system alert policies and procedures   True   
1432   K1272                     Knowledge of system components   True   
1433   K1273  Knowledge of threat investigation policies and...   True   
1434   K1274  Knowledge of threat modeling tools and techniq

In [91]:
print(kds.columns)
kds = kds.drop(columns=['New', 'Withdrawn', 'Replacement Statements (if applicable)'])

Index(['KSAT ID', 'Statement Description', 'New', 'Withdrawn',
       'Replacement Statements (if applicable)'],
      dtype='object')


In [92]:
print(kds.columns)

Index(['KSAT ID', 'Statement Description'], dtype='object')


In [93]:
# index the descriptions. 
kds_old = kds[kds['KSAT ID'].isin(KD_index)]
print(kds_old)

    KSAT ID                              Statement Description
177   K0001  Knowledge of computer networking concepts and ...
178   K0002  Knowledge of risk management processes (e.g., ...
179   K0003  Knowledge of laws, regulations, policies, and ...
180   K0004  Knowledge of cybersecurity and privacy principles
181   K0005     Knowledge of cyber threats and vulnerabilities
..      ...                                                ...
790   K0614  Knowledge of wireless technologies (e.g., cell...
791   K0615  Knowledge of privacy disclosure statements bas...
798   K0622  Knowledge of controls related to the use, proc...
800   K0624  Knowledge of Application Security Risks (e.g. ...
804   K0628  Knowledge of cyber competitions as a way of de...

[576 rows x 2 columns]


In [100]:
df.rename(columns={"knowledge descriptions(NICE Framework)":"KSAT ID"}, inplace=True)
df = df.merge(kds_old, on="KSAT ID", how='left')
print(df.columns)
print(kds_old.columns)

Index(['KSAT ID', '0', '1', '2', '3', '4', '5', '6', '7', '8',
       'Statement Description'],
      dtype='object')
Index(['KSAT ID', 'Statement Description'], dtype='object')


In [102]:
print(df)
print(df.columns)

    KSAT ID  0  1  2  3  4  5  6  7  8  \
0     K0001  0  0  0  0  1  0  0  0  0   
1     K0002  0  0  0  0  0  0  0  1  0   
2     K0003  0  0  0  0  0  0  0  0  1   
3     K0004  0  0  1  0  0  1  0  0  1   
4     K0005  1  0  0  1  1  1  0  0  0   
..      ... .. .. .. .. .. .. .. .. ..   
571   K0614  1  0  0  0  1  0  0  0  0   
572   K0615  0  0  0  0  0  0  1  0  0   
573   K0622  0  0  0  0  0  0  0  1  0   
574   K0624  1  0  1  0  0  0  0  0  0   
575   K0628  1  0  0  0  0  0  0  0  0   

                                 Statement Description  
0    Knowledge of computer networking concepts and ...  
1    Knowledge of risk management processes (e.g., ...  
2    Knowledge of laws, regulations, policies, and ...  
3    Knowledge of cybersecurity and privacy principles  
4       Knowledge of cyber threats and vulnerabilities  
..                                                 ...  
571  Knowledge of wireless technologies (e.g., cell...  
572  Knowledge of privacy disclosure st

In [103]:
# now save in the desired format (pytorch tensor?) for processing by huggingface
from datasets import Dataset 
dataset = Dataset.from_pandas(df)
print(dataset)

Dataset({
    features: ['KSAT ID', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description'],
    num_rows: 576
})


In [104]:
# save to disk. 
dataset.save_to_disk("data/train.hf")

Saving the dataset (0/1 shards):   0%|          | 0/576 [00:00<?, ? examples/s]

In [5]:
# load from disk. 
from datasets import load_from_disk
ds = load_from_disk("data/train.hf")

In [6]:
print(ds)

Dataset({
    features: ['KSAT ID', '0', '1', '2', '3', '4', '5', '6', '7', '8', 'Statement Description'],
    num_rows: 576
})
